<a href="https://colab.research.google.com/github/fc63/gender-classification/blob/main/rus_translate/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers sentencepiece pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import pickle
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

input_path = "/content/drive/MyDrive/datasets/rus_gender.pkl"

with open(input_path, "rb") as f:
    df = pickle.load(f)

# İlk satırları göster
print(df.head())
# Veri hakkında genel bilgi
print(df.info())

                                                text gender
0  здравствуйте. я покупал у вас путевку в тур. я...   male
1  Всем приветики ) Меня зовут Лена, На этом сайт...   male
2  Уважаемый начальник. Принимая во внимание вашу...   male
3  Приветик горячий парень с огненными глазами!Тв...   male
4  Добрый день! Мне надоело ваше отношение ко мне...   male
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9564 entries, 0 to 9563
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    9564 non-null   object
 1   gender  9564 non-null   object
dtypes: object(2)
memory usage: 149.6+ KB
None


In [ ]:
df

,text,gender
0,здравствуйте. я покупал у вас путевку в тур. я...,male
1,"Всем приветики ) Меня зовут Лена, На этом сайт...",male
2,Уважаемый начальник. Принимая во внимание вашу...,male
3,Приветик горячий парень с огненными глазами!Тв...,male
4,Добрый день! Мне надоело ваше отношение ко мне...,male
...,...,...
9559,Привет Серёга. Хочешь оттянуться на этих выход...,male
9560,"Доброго здравия, Евгений Абрамович. Как прожив...",male
9561,"привет, для продолжения сотрудничества пре...",male
9562,"здарова, погнали чилить в рестик, там и побаза...",male


In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer

# GPU varsa kullan
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Rusça-İngilizce modeli
model_name = "Helsinki-NLP/opus-mt-ru-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/803k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/307M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:
def translate_batch(texts, batch_size=8):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model.generate(
            **inputs,
            num_beams=2,
            no_repeat_ngram_size=4
        )
        translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(translations)
    return results

ru_texts = df['text'].astype(str).tolist()
en_texts = translate_batch(ru_texts)

# Yeni DataFrame oluşturma
translated_df = pd.DataFrame({
    'gender': df['gender'].tolist(),
    'text': en_texts
})

translated_df.head()

model.safetensors:   0%|          | 0.00/307M [00:00<?, ?B/s]

,gender,text
0,male,Hey. I bought a trip from you on tour. I'm ver...
1,male,"My name is Lena, on this website I'm going to ..."
2,male,"With your leadership in our team, I can't igno..."
3,male,"Hello, hot guy with eyes of fire! Your eyes ha..."
4,male,"I'm sick of your attitude to me during work, y..."


In [ ]:
translated_df

,gender,text
0,male,Hey. I bought a trip from you on tour. I'm ver...
1,male,"My name is Lena, on this website I'm going to ..."
2,male,"With your leadership in our team, I can't igno..."
3,male,"Hello, hot guy with eyes of fire! Your eyes ha..."
4,male,"I'm sick of your attitude to me during work, y..."
...,...,...
9559,male,"Hey, Seryoga, you want to hang out this weeken..."
9560,male,"Good health, Evgeny Abramovich. How does your ..."
9561,male,"Hi, for further cooperation, I suggest we meet..."
9562,male,"Here, let's get some chili in the restica, and..."


In [ ]:
with open("/content/drive/MyDrive/datasets/rus_gender_en.pkl", "wb") as f:
    pickle.dump(translated_df, f)